# Data-Efficient Vision Transformers for Small Image Datasets
### IT3071 Machine Learning — Explainer Notebook

This notebook walks through our full experiment end-to-end.

**The core question:** *Can a Vision Transformer (ViT) compete with a CNN when trained on a small dataset?*

**Short answer:** Not from scratch — but if we fine-tune a pretrained ViT, it wins easily.

**What we compare:**
| Model | Training | Expected result |
|---|---|---|
| ViT-Tiny (from scratch) | Random init, trained only on our small dataset | Worst — no inductive bias, no pretraining |
| ResNet-18 (CNN) | Random init | Mid — convolutional inductive bias helps |
| ViT-Tiny (pretrained) | ImageNet weights, fine-tuned | Best — pretraining compensates for small data |

**Why this ranking?** CNNs bake in two assumptions about images: *locality* (nearby pixels are related) and *translation invariance* (a cat is still a cat if you shift it left). ViTs make neither assumption — they learn these patterns from data. With enough data that's a strength (they can learn richer patterns). With a small dataset it's a weakness.

Run the cells top-to-bottom. GPU recommended for the training cells.

## 1. Setup

Install dependencies and import all modules. The `sys.path` insert lets us import from `src/` whether this notebook is run from the repo root or from inside `notebooks/`.

In [ ]:
# On Google Colab: clone the repo first, then run this cell.
# !git clone https://github.com/Dippy2003/data-efficient-vit.git
# %cd data-efficient-vit
# !pip install -r requirements.txt

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "") or "..")

import random
import numpy as np
import torch
import matplotlib.pyplot as plt

# Fix random seeds so results are reproducible across runs
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

from src.data      import get_dataloaders, CIFAR10_CLASSES
from src.models    import build_model, count_parameters, MODEL_REGISTRY
from src.train     import get_device, train_model, load_checkpoint, DEV_CONFIG, FINAL_CONFIG
from src.evaluate  import (compute_accuracy, confusion_matrix_from_loader,
                           per_class_report, build_results_table, print_results_table)
from src.visualize import (plot_training_curves, plot_confusion_matrix,
                           plot_sample_predictions, plot_attention_overlay)

DEVICE = get_device()
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

---
## 2. What is a Vision Transformer?

A **Vision Transformer (ViT)** applies the Transformer architecture — originally designed for text — to images.

**How it works (step by step):**
1. **Patch embedding** — The image is split into a grid of fixed-size patches (16×16 pixels each). Each patch is flattened and linearly projected into a vector called a *token*, just like a word embedding in NLP.
2. **Positional encoding** — A learnable position embedding is added to each token so the model knows *where* each patch came from in the image.
3. **Transformer encoder** — The sequence of patch tokens is fed through stacked Multi-Head Self-Attention + Feed-Forward layers. Each token can attend to every other token equally, learning which patches are important to each other.
4. **Classification head** — A special `[CLS]` token is prepended; its final representation is passed through a linear layer to produce class logits.

**Why ViTs struggle on small datasets:**
- Self-attention is *global* — every patch attends to every other patch from the start. There is no built-in assumption that nearby patches are more related than distant ones.
- CNNs use *convolution kernels* that enforce locality by design: each neuron only looks at a small neighbourhood. This is a strong prior that turns out to be correct for most natural images.
- When training data is scarce, priors matter a lot. The CNN gets spatial priors for free; the ViT has to discover them from examples — and with too few examples, it never quite does.

> **Key takeaway:** The from-scratch ViT and the CNN have similar numbers of parameters (~5M vs ~11M), so the performance gap is about *architecture*, not *size*.

---
## 3. Data Pipeline

We use **CIFAR-10**: 60,000 32×32 colour images across 10 classes (50,000 train / 10,000 test). It's a classic benchmark that's small enough to train on consumer hardware, but large enough to show meaningful differences between models.

**Key pipeline choices:**
- Images are resized to **224×224** even though CIFAR-10 is natively 32×32. This is necessary because the pretrained ViT's position embeddings were trained at 224×224 resolution (matching ImageNet). Using a smaller size would break the positional encoding.
- **Augmentation** (random crop + horizontal flip) is applied only to the training set — validation/test use the clean image so metrics reflect real performance.
- `subset_fraction` controls how much training data to use. **0.05 during development** (fast), **1.0 for the final reported run**.

> Change `USE_FULL_DATA = False` → `True` before your final run.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
USE_FULL_DATA = False          # set True for the final graded run
IMAGE_SIZE    = 224
BATCH_SIZE    = 64
# ─────────────────────────────────────────────────────────────────────────────

config = FINAL_CONFIG if USE_FULL_DATA else DEV_CONFIG

loaders = get_dataloaders(
    dataset_name    = "cifar10",
    data_root       = "../data",
    image_size      = IMAGE_SIZE,
    batch_size      = BATCH_SIZE,
    subset_fraction = config["subset_fraction"],
    num_workers     = 0,          # set to 2 on Linux/Colab for faster loading
)

print(f"Classes : {CIFAR10_CLASSES}")
print(f"Train   : {len(loaders['train'].dataset)} samples")
print(f"Val     : {len(loaders['val'].dataset)} samples")
print(f"Test    : {len(loaders['test'].dataset)} samples")

### Sample images from CIFAR-10

Let's visualise one example per class to see what the model is working with. Note how similar some classes look at 32×32 (cat vs dog, automobile vs truck) — these are the pairs we expect to see confused in the confusion matrix later.

In [ ]:
CIFAR10_MEAN = np.array([0.4914, 0.4822, 0.4465])
CIFAR10_STD  = np.array([0.2470, 0.2435, 0.2616])

def denorm(tensor):
    """Undo the ImageNet-style normalisation so we can display the image."""
    img = tensor.numpy().transpose(1, 2, 0)
    return np.clip(img * CIFAR10_STD + CIFAR10_MEAN, 0, 1)

# Collect one example per class from the test loader (no augmentation)
seen = {}
for images, labels in loaders["test"]:
    for img, lbl in zip(images, labels):
        cls = lbl.item()
        if cls not in seen:
            seen[cls] = img
        if len(seen) == 10:
            break
    if len(seen) == 10:
        break

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(denorm(seen[i]))
    ax.set_title(CIFAR10_CLASSES[i], fontsize=11)
    ax.axis("off")

fig.suptitle("CIFAR-10 — one example per class", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 4. Model Architectures

We build three models using `src/models.py`'s `build_model()` factory.

**ViT-Tiny (from scratch)** — `vit_scratch`
- Architecture: 12 transformer blocks, 3 attention heads, embedding dim 192, patch size 16×16
- Initialisation: random — must learn *everything* from our small dataset
- Why "tiny"? The smallest standard ViT variant, keeping training time reasonable

**ResNet-18 (CNN)** — `cnn`
- Architecture: 8 residual blocks with skip connections, convolutions throughout
- Initialisation: random — same as ViT-scratch, fair comparison
- Why ResNet-18? A well-understood, widely-used baseline that isn't too large

**ViT-Tiny (pretrained)** — `vit_pretrained`
- Same architecture as ViT-scratch
- Initialisation: **ImageNet-21k pretrained weights** (loaded via `timm`)
- Why pretrained helps: the model already knows edges, textures, object parts — fine-tuning just adapts these to our 10 classes

> Notice that ViT-scratch has **fewer parameters than ResNet-18**. Any accuracy gap between them is about inductive bias, not capacity.

In [ ]:
print(f"{'Model':<20} {'Total params':>15} {'Trainable':>15}")
print("-" * 52)
for name in MODEL_REGISTRY:
    m = build_model(name, num_classes=10, img_size=IMAGE_SIZE)
    p = count_parameters(m)
    print(f"{name:<20} {p['total']:>15,} {p['trainable']:>15,}")